# 타이타닉 분석 검증 노트북

앞선 분석의 결론이 정말 믿을 만한지 **스스로 반박**해 보는 노트북입니다.

점검 항목:
1. 데이터 무결성·출처 검증
2. 기준선(baseline)과 지표 재정의
3. 교차검증 + 신뢰구간
4. 임계값·정밀도/재현율 분석
5. Age 비선형 관계 검증
6. 가족/그룹 효과와 표본 불확실성
7. 다중비교 경고

> 결론: 정확도 하나만 보면 과대평가됩니다. 아래를 실행하면 대부분의 세부 수치가 **노이즈 범위**임을 확인할 수 있습니다.

## 0. 준비

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="AppleGothic")
RANDOM_STATE = 42

In [ ]:
DATA_PATH = "/Users/remchoi/.cache/kagglehub/datasets/heptapod/titanic/versions/1/train_and_test2.csv"
raw = pd.read_csv(DATA_PATH).rename(columns={"2urvived": "Survived"})
df = raw.drop(columns=[c for c in raw.columns if c.startswith("zero")] + ["Passengerid"]).copy()
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df.head()

## 1. 데이터 무결성·출처 검증

여기서 확인할 것:

- 행 수가 원본 train(891) + test(418)인지
- `zero.*` 상수 컬럼이 있는지
- Age 결측은 없는데 **소수점 나이**가 있는지 (인위적 대체 흔적)
- `Fare == 0` 이 진짜 무료인지, 결측을 0으로 코딩한 것인지
- Sex 인코딩(0/1)이 실제로 무엇인지, 생존율이 알려진 타이타닉과 맞는지

In [ ]:
zero_cols = [c for c in raw.columns if c.startswith("zero")]
print("전체 행 수:", len(raw), "| 891 + 418 =", 891 + 418)
print("zero.* 상수 컬럼 수:", len(zero_cols))
print("Age 결측 수:", raw["Age"].isnull().sum())
print("Age 소수점 값 개수:", (df["Age"] % 1 != 0).sum())
print("가장 흔한 Age 값:", df["Age"].mode().tolist(), "| Age==28 인원:", (df["Age"] == 28).sum())
print()
print("Sex 분포:")
print(df["Sex"].value_counts())
print()
print("Sex 별 생존율 (알려진 타이타닉: 여성 약 0.74, 남성 약 0.19):")
print(df.groupby("Sex")["Survived"].agg(["mean", "count"]).round(3))

In [ ]:
fzero = df[df["Fare"] == 0]
print("Fare==0 인원:", len(fzero))
print("그중 Sex 분포:", fzero["Sex"].value_counts().to_dict())
print("그중 Embarked 분포:", fzero["Embarked"].value_counts().to_dict())
print("그중 Age==28 인원:", (fzero["Age"] == 28).sum())
print("그중 생존율:", round(fzero["Survived"].mean(), 3))
fzero[["Sex", "Pclass", "Age", "sibsp", "Parch", "Embarked", "Survived"]]

**해석**: Age 결측이 0인데 소수점 나이가 존재하고 특정 값(28)이 반복됩니다. `Fare==0` 17명도
모두 남성·동일 항구이며 상당수가 Age=28입니다. 이는 **결측을 0/대표값으로 대체한 흔적**으로,
`Fare`·`Age`를 원값처럼 다루면 왜곡됩니다. 또한 여성 생존율 0.50은 알려진 0.74와 크게 다릅니다.

## 2. 기준선(baseline)과 지표 재정의

생존율이 26%뿐이라 "전부 사망"으로 찍어도 정확도가 높습니다.
정확도만 보면 모델을 과대평가하므로 **재현율(recall)** 을 함께 봐야 합니다.

In [ ]:
baseline = 1 - df["Survived"].mean()
print(f"'전부 사망' 기준선 정확도: {baseline:.3f}")
print(f"생존율(양성 비율): {df['Survived'].mean():.3f}")

## 3. 교차검증 + 신뢰구간

한 번의 train/test 분할 대신 **5-겹 교차검증**으로 평균과 표준편차를 구합니다.
`mean ± 1.96 * std / sqrt(k)` 로 95% 신뢰구간을 근사합니다.

In [ ]:
X = df.drop(columns="Survived")
y = df["Survived"]

models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy": "accuracy", "roc_auc": "roc_auc", "recall": "recall"}

rows = []
for name, model in models.items():
    for metric, scorer in scoring.items():
        scores = cross_val_score(model, X, y, cv=cv, scoring=scorer)
        rows.append([name, metric, scores.mean(), scores.std(),
                     scores.mean() - 1.96 * scores.std() / np.sqrt(len(scores)),
                     scores.mean() + 1.96 * scores.std() / np.sqrt(len(scores))])

cv_table = pd.DataFrame(rows, columns=["모델", "지표", "평균", "표준편차", "CI하한", "CI상한"])
print(cv_table.round(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=cv_table[cv_table["지표"].isin(["accuracy", "roc_auc"])],
            x="모델", y="평균", hue="지표", ax=ax, palette="Set2")
ax.axhline(baseline, color="red", linestyle="--", label=f"기준선 {baseline:.3f}")
ax.set_title("교차검증 평균 점수 (오차막대=표준편차)")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
acc = cv_table[(cv_table["지표"] == "accuracy")]
print("정확도 95% CI가 기준선을 넘는 모델:")
print(acc[acc["CI하한"] > baseline][["모델", "평균", "CI하한", "CI상한"]].round(3).to_string(index=False))
print()
print("정확도 CI가 서로 겹치면 '더 좋다'고 말할 수 없습니다. 위 표의 CI하한/상한을 비교하세요.")

## 4. 임계값·정밀도/재현율 분석

분류는 확률 0.5를 기준으로 자릅니다. 임계값을 바꾸면 정확도·재현율이 크게 달라집니다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=RANDOM_STATE)
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]

rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (proba >= t).astype(int)
    tp = ((pred == 1) & (y_test == 1)).sum()
    fn = ((pred == 0) & (y_test == 1)).sum()
    fp = ((pred == 1) & (y_test == 0)).sum()
    recall = tp / (tp + fn) if (tp + fn) else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    rows.append([t, accuracy_score(y_test, pred), precision, recall, f1])

threshold_table = pd.DataFrame(rows, columns=["임계값", "정확도", "정밀도", "재현율", "F1"])
print(threshold_table.round(3).to_string(index=False))

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, proba)
plt.figure(figsize=(7, 5))
plt.plot(recall, precision, marker=".")
plt.axvline(threshold_table.loc[threshold_table["임계값"] == 0.5, "재현율"].values[0],
            color="red", linestyle="--", label="임계값 0.5의 재현율")
plt.xlabel("재현율 (Recall)")
plt.ylabel("정밀도 (Precision)")
plt.title("Precision-Recall 곡선")
plt.legend()
plt.tight_layout()
plt.show()

**해석**: 임계값 0.5에서 재현율이 약 0.45로, **실제 생존자의 절반 이상을 놓칩니다**.
정확도 0.76은 "대부분을 사망이라 찍어서" 나온 착시입니다.

## 5. Age 비선형 관계 검증

상관계수는 직선 관계만 잡습니다. 나이를 구간으로 나눠 생존율을 보면 '어린이 우선' 패턴이 드러납니다.
Wilson 신뢰구간으로 표본 불확실성도 함께 표시합니다.

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0, 0)
    p = k / n
    d = 1 + z**2 / n
    center = p + z**2 / (2 * n)
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
    return ((center - margin) / d, (center + margin) / d)

bins = [0, 5, 12, 18, 30, 45, 60, 100]
df["AgeBin"] = pd.cut(df["Age"], bins=bins)

age_stats = df.groupby("AgeBin", observed=True)["Survived"].agg(["sum", "count"])
age_stats["생존율"] = age_stats["sum"] / age_stats["count"]
age_stats[["CI하한", "CI상한"]] = age_stats.apply(
    lambda r: pd.Series(wilson_ci(r["sum"], r["count"])), axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(age_stats))
ax.bar(x, age_stats["생존율"], color="steelblue", alpha=0.7)
ax.errorbar(x, age_stats["생존율"],
            yerr=[np.clip(age_stats["생존율"] - age_stats["CI하한"], 0, None),
                  np.clip(age_stats["CI상한"] - age_stats["생존율"], 0, None)],
            fmt="none", color="black", capsize=4)
ax.set_xticks(list(x))
ax.set_xticklabels([str(i) for i in age_stats.index], rotation=30)
ax.set_title("나이 구간별 생존율 (오차막대=95% Wilson CI)")
ax.set_ylabel("생존율")
plt.tight_layout()
plt.show()

print("Age 상관계수(Pearson):", round(df["Age"].corr(df["Survived"]), 3))
age_stats[["count", "생존율", "CI하한", "CI상한"]].round(3)

**해석**: 전상관은 -0.06에 불과하지만, 0~5세 생존율은 약 0.55로 18~30세(0.22)보다 훨씬 높습니다.
즉 **"나이 효과 없음"이 아니라 비선형 관계**이며, 상관계수만 본 1차 분석은 이 사실을 놓쳤습니다.

## 6. 가족/그룹 효과와 표본 불확실성

- `FamilySize` 별 생존율을 Wilson CI와 함께 봅니다. 표본이 적은 구간의 CI가 얼마나 넓은지 확인합니다.
- 같은 가족은 운명을 공유하므로 행이 독립이 아닙니다. 티켓 정보가 없어
  `(Pclass, Fare, Embarked, sibsp, Parch)` 조합을 **가족 대리 그룹**으로 삼아 `GroupKFold` 를 돌려봅니다.

In [ ]:
df["FamilySize"] = df["sibsp"] + df["Parch"] + 1
fam = df.groupby("FamilySize")["Survived"].agg(["sum", "count"])
fam["생존율"] = fam["sum"] / fam["count"]
fam[["CI하한", "CI상한"]] = fam.apply(lambda r: pd.Series(wilson_ci(r["sum"], r["count"])), axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(fam.index, fam["생존율"], color="mediumseagreen", alpha=0.7)
ax.errorbar(fam.index, fam["생존율"],
            yerr=[np.clip(fam["생존율"] - fam["CI하한"], 0, None),
                  np.clip(fam["CI상한"] - fam["생존율"], 0, None)],
            fmt="none", color="black", capsize=4)
ax.set_title("FamilySize 별 생존율 (오차막대=95% Wilson CI)")
ax.set_xlabel("FamilySize")
ax.set_ylabel("생존율")
plt.tight_layout()
plt.show()

fam[["count", "생존율", "CI하한", "CI상한"]].round(3)

In [ ]:
group_key = df[["Pclass", "Fare", "Embarked", "sibsp", "Parch"]].astype(str).agg("|".join, axis=1)
groups = group_key.factorize()[0]
print("대리 그룹 수:", len(set(groups)), "| 승객 수:", len(groups))
print("한 그룹 최대 인원:", pd.Series(groups).value_counts().max())

gcv = GroupKFold(n_splits=5)
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=gcv, groups=groups, scoring="roc_auc")
    print(f"{name:20s} GroupKFold AUC 평균={scores.mean():.3f} (표준편차={scores.std():.3f})")

**해석**: 표본이 적은 가족 규모(8, 11 등)는 95% CI가 [0, 0.32]처럼 매우 넓어
"대가족은 죽는다"고 단정할 수 없습니다. 다만 GroupKFold AUC가 일반 CV와 비슷하게 나오면
(RandomForest 약 0.79), 가족 중복으로 점수가 크게 부풀려졌다고 보기는 어렵습니다.

## 7. 다중비교 경고

변수마다 상관계수를 구하고 여러 그래프를 그리면, 우연히 나타나는 패턴도 섞입니다.
부트스트랩으로 생존 상관계수의 신뢰구간을 구해 **0을 포함하는지** 봅니다.

In [ ]:
def bootstrap_corr(x, y, n_boot=2000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    out = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        out[i] = np.corrcoef(x[idx], y[idx])[0, 1]
    return np.percentile(out, [2.5, 97.5])

num_cols = ["Sex", "Pclass", "Fare", "Age", "FamilySize", "sibsp", "Parch", "Embarked"]
rows = []
for col in num_cols:
    r = df[col].corr(df["Survived"])
    lo, hi = bootstrap_corr(df[col], df["Survived"])
    rows.append([col, r, lo, hi, "0 포함(불확실)" if lo <= 0 <= hi else "0 미포함"])

boot_table = pd.DataFrame(rows, columns=["변수", "상관계수", "CI하한", "CI상한", "판정"])
print(boot_table.round(3).to_string(index=False))

**해석**: `FamilySize`(+0.02)와 `sibsp`(-0.01)는 95% CI가 0을 포함해 **우연과 구분 불가**입니다.
`Age`(-0.113~-0.001)와 `Parch`(0.004~0.116)는 0에 아주 근접한 경계선입니다.
반면 `Sex`(+0.35~+0.46), `Pclass`(-0.30~-0.19), `Fare`(+0.12~+0.23)는 0을 포함하지 않아
상대적으로 견고합니다. (`Embarked`도 0을 포함하지 않지만 Pclass와 교란됨)

## 최종 정리: 무엇을 믿고 무엇을 버릴까

| 주장 | 신뢰도 | 이유 |
|---|---|---|
| `Sex`, `Pclass` 가 생존과 연관 | **중간~높음** | CV·부트스트랩 CI 모두 0 미포함 |
| `Fare` 가 생존과 연관 | 중간 | Pclass와 -0.56 공선성, 대리 변수 가능성 |
| 모델 정확도 0.76이 좋은 성능 | **낮음** | 기준선 0.739, 재현율 0.46, CI 넓음 |
| `FamilySize` 비선형 효과 | 낮음 | 소표본, 가족 중복, CI가 0 포함 |
| `Age` 는 무관 | **틀림** | 0~5세 생존율 0.55로 비선형 |
| 특정 모델이 최고 | 알 수 없음 | 모델 간 차이가 CI 내 노이즈 |

### 재현 가능한 결론
1. 이 데이터는 **2차 가공본**(행 수 병합, 상수 컬럼, 인위적 결측 대체)이라 원본 타이타닉과 다릅니다.
2. 정확도 단독 보고는 금물이며, **기준선·재현율·교차검증 CI**를 함께 봐야 합니다.
3. 견고한 신호는 `Sex`·`Pclass` 정도이고, 나머지 세부 수치는 탐색적 가설로만 다루어야 합니다.

### 다음 단계
- 가족/티켓 그룹 단위 교차검증 강화
- Age 구간·상호작용 항을 넣은 모델 재학습
- `Fare==0`·결측 대체 규칙 원본 확인 후 재현